# 03 Build the CNN patch dataset  (optional extra; requires GEE exports and raw data)

The small CNN is an extra of the paper; the classical models (notebooks 04 to 06) do not need this notebook.

Reads the Sentinel-1/2 patches exported by `gee/export_frantz_features.js` (`PRODUCT = 'patches'`), adds three dense
target layers computed from the footprints and the nDSM (`ndsm_mean`, `bldg_height`, `bldg_frac`) and writes one
TFRecord per city. Needs the raw data in `data_raw/` and `pip install tensorflow` (not part of `environment.yml`).

Run it twice, with `SET = "train"` and `SET = "test"` (natural test patches). Dataset format and loader:
`docs/README_patch_dataset.md`. The CNN training code is not part of this repository.

In [ ]:
# Setup: make `config.py` importable whether Jupyter was started in the repo root or in notebooks/
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
for _p in (_here, _here / "notebooks", _here.parent / "notebooks"):
    if (_p / "config.py").exists():
        sys.path.insert(0, str(_p)); break
else:
    raise FileNotFoundError("config.py not found: start Jupyter from the repository root or from notebooks/")
import config
print("repo root:", config.ROOT)

In [ ]:
# ---------------------------------------------------------------------
# Dense-target patch dataset writer, 8 cities (train or natural TEST patches).
# Notebook cell. Run it twice: SET = "train", then SET = "test".
#
# Per city it does ONE of two things:
#  * BUILD: the city has no output file yet (the five new cities). Reads that city's GEE patch
#    TFRecords, adds the three dense target layers from the footprints and the nDSM, writes
#    <city>_dataset.tfrecord.gz (train) or <city>_test_dataset.tfrecord.gz (test).
#  * RELABEL: the output file already exists (Porto Alegre, Sao Paulo, Rio de Janeiro). Only the
#    `split` field of every record is rewritten from the features table; x and y are copied unchanged,
#    so the slow target computation is not repeated. This is how the rows the rebalancing left
#    out get split = "unused" (and the 50 m exclusions get "excluded").
#    REBUILD = ["porto_alegre"] forces a full BUILD for a city instead.
#
# split values after this cell: train | excluded | unused (training files), test (test files).
# The loader keeps only "train" for training and validation, so "unused" and "excluded" drop out.
#
# Cities are processed one at a time (memory); each city's GEE files are found by the
# <city>_patches_c*.tfrecord.gz prefix in any of the SRC folders.
# ---------------------------------------------------------------------
import glob, os, numpy as np, pandas as pd, tensorflow as tf, rasterio, geopandas as gpd
from rasterio.features import rasterize
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.windows import Window
from shapely.geometry import box
from pyproj import Transformer

SET = "train"                                              # "train" | "test": run the notebook once with each
REBUILD = []                                                # cities to BUILD even if their output exists


# SPLIT_TABLE: the shipped 8-city table, read with config.read_table()
FOOTPRINTS = config.FOOTPRINTS
NDSM = config.NDSM
MIN_PART_AREA_M2 = globals().get("MIN_PART_AREA_M2", 10.0)

assert SET in ("train", "test"), 'SET must be "train" or "test"'
# GEE export folders on Drive; add the folder of the new-city exports here if it has another name
SRC = [config.GEE_PATCHES_NATURAL if SET == "test" else config.GEE_PATCHES]
OUT = os.path.join(config.OUT, "patch_dataset" + ("_natural" if SET == "test" else ""))
FNAME = (lambda city: f"{city}_test_dataset.tfrecord.gz") if SET == "test" else (lambda city: f"{city}_dataset.tfrecord.gz")
BLOCK_M = 2560.0
# UTM zone (WGS84) of each city's patch grid. MUST be the CRS the GEE exporter used; the guard below checks it.
CRS = {"porto_alegre": "EPSG:32722", "curitiba": "EPSG:32722",
       "sao_paulo": "EPSG:32723", "rio_de_janeiro": "EPSG:32723", "niteroi": "EPSG:32723", "belo_horizonte": "EPSG:32723",
       "salvador": "EPSG:32724", "recife": "EPSG:32725"}
S, R, PX, NODATA = 33, 16, 10, -9999.0
BANDS = ["VV", "VH", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
NAMES = [f"S{s}_{b}" for s in range(1, 5) for b in BANDS]
FLIP_Y = True                                               # EE arrays arrive south-up; output is north-up
TARGETS = ["ndsm_mean", "bldg_height", "bldg_frac"]
TRAIN_SPLITS = {"train", "excluded", "unused"}


def corners(x, y):
    x0, y0 = np.floor(x / PX) * PX, np.floor(y / PX) * PX
    west, north = x0 - R * PX, y0 + PX + R * PX
    return west, north - S * PX, west + S * PX, north


def target_layers(src, fp, x, y, crs):
    west, south, east, north = corners(x, y)
    dst_tf = from_origin(west, north, PX, PX)
    l, b, r, t = transform_bounds(crs, src.crs, west, south, east, north)
    w0 = src.window(l, b, r, t)
    win = Window(int(np.floor(w0.col_off)) - 1, int(np.floor(w0.row_off)) - 1,
                 int(np.ceil(w0.width)) + 3, int(np.ceil(w0.height)) + 3)
    nod = src.nodata if src.nodata is not None else NODATA
    arr = src.read(1, window=win, boundless=True, fill_value=nod).astype("float64")
    wt = src.window_transform(win)
    valid = np.isfinite(arr) & (arr != nod)
    arr = np.where(valid, np.maximum(arr, 0.0), 0.0).astype("float32")
    idx = fp.sindex.query(box(l, b, r, t))
    bld = (rasterize([(g, 1) for g in fp.geometry.iloc[idx]], out_shape=arr.shape, transform=wt,
                     fill=0, dtype="uint8").astype(bool) if len(idx) else np.zeros(arr.shape, bool))
    wgt = (bld & valid).astype("float32")
    def avg(a):
        out = np.zeros((S, S), "float32")
        reproject(a.astype("float32"), out, src_transform=wt, src_crs=src.crs,
                  dst_transform=dst_tf, dst_crs=crs, resampling=Resampling.average)
        return out
    with np.errstate(divide="ignore", invalid="ignore"):
        allv = avg(valid); den = avg(wgt)
        mean_all = np.where(allv > 0, avg(arr) / allv, NODATA)
        mean_b = np.where(den > 0, avg(arr * wgt) / den, NODATA)
    return np.stack([mean_all, mean_b, avg(bld)]).astype("float32")


spec = {n: tf.io.FixedLenFeature([S * S], tf.float32) for n in NAMES}
spec.update({k: tf.io.FixedLenFeature([1], tf.float32) for k in ["height_m", "is_background", "x_m", "y_m"]})
spec.update({k: tf.io.FixedLenFeature([1], tf.string) for k in ["sample_id", "city"]})
fl = lambda v: tf.train.Feature(float_list=tf.train.FloatList(value=np.asarray(v, "float32").ravel()))
by = lambda v: tf.train.Feature(bytes_list=tf.train.BytesList(value=[v]))
GZ = tf.io.TFRecordOptions(compression_type="GZIP")

# ---- split table: which ids exist, their split, and how many records to expect per city ----
st = config.read_table(usecols=["sample_id", "city", "split"])
split_of = dict(zip(st.sample_id, st.split))
want = st[st.split.isin(TRAIN_SPLITS)] if SET == "train" else st[st.split == "test"]
EXPECTED = want.groupby("city").size().to_dict()
print(f"split table: {len(st)} ids | {st.split.value_counts().to_dict()}")
print(f"expected {SET} records per city: {EXPECTED}")
missing_crs = set(EXPECTED) - set(CRS)
if missing_crs:
    raise ValueError(f"no CRS set for {sorted(missing_crs)}")


def check_ids(city, ids):
    nat = sum("_nat_" in i for i in ids)
    if (SET == "test") != (nat == len(ids)) or nat not in (0, len(ids)):
        raise ValueError(f"{city}: {nat} of {len(ids)} ids contain '_nat_', which does not fit SET = {SET}")
    if len(set(ids)) != len(ids):
        raise ValueError(f"{city}: {len(ids) - len(set(ids))} duplicated sample_ids (a task exported twice?)")
    miss = [i for i in ids if i not in split_of]
    if miss:
        raise ValueError(f"{city}: {len(miss)} ids not in the features table, e.g. {miss[:3]}")
    bad = [i for i in ids if (split_of[i] == "test") != (SET == "test")]
    if bad:
        raise ValueError(f"{city}: {len(bad)} ids have a split that does not fit SET = {SET}, e.g. {bad[:3]} -> {split_of[bad[0]]}")
    if len(ids) != EXPECTED.get(city, 0):
        print(f"  WARNING {city}: {len(ids)} records, table expects {EXPECTED.get(city, 0)} "
              f"(missing export task, or ids in the table without a patch)")


def relabel(city, path):
    """Rewrite only the `split` field of an existing output file (x, y and the rest are copied unchanged)."""
    tmp = path + ".tmp"
    n, ids, old_new = 0, [], {}
    with tf.io.TFRecordWriter(tmp, GZ) as wtr:
        for raw in tf.data.TFRecordDataset([path], compression_type="GZIP").as_numpy_iterator():
            ex = tf.train.Example.FromString(raw)
            sid = ex.features.feature["sample_id"].bytes_list.value[0].decode()
            ids.append(sid)
            old = ex.features.feature["split"].bytes_list.value[0].decode() if "split" in ex.features.feature else "none"
            new = split_of.get(sid, "MISSING")
            old_new[(old, new)] = old_new.get((old, new), 0) + 1
            ex.features.feature["split"].CopyFrom(by(new.encode()))
            wtr.write(ex.SerializeToString()); n += 1
    try:
        check_ids(city, ids)
    except ValueError:
        os.remove(tmp); raise
    os.replace(tmp, path)
    print(f"{city}: RELABEL {n} records in {path}")
    print("  old -> new split:", {f"{o} -> {nw}": c for (o, nw), c in sorted(old_new.items())})


def build(city, out_path):
    files = sorted(set(f for d in SRC for f in glob.glob(os.path.join(d, f"{city}_patches_c*.tfrecord.gz"))))
    if not files:
        print(f"{city}: no GEE files {city}_patches_c*.tfrecord.gz in {SRC}; skipped")
        for d in SRC:
            print("   ", d, "exists" if os.path.isdir(d) else "MISSING",
                  sorted(os.listdir(d))[:20] if os.path.isdir(d) else "")
        return
    print(f"{city}: BUILD from {len(files)} files ({sum(os.path.getsize(f) for f in files)/1e6:.0f} MB)")
    exs = list(tf.data.TFRecordDataset(files, compression_type="GZIP")
               .map(lambda r: tf.io.parse_single_example(r, spec)).as_numpy_iterator())
    ids = [e["sample_id"][0].decode() for e in exs]
    check_ids(city, ids)
    crs = CRS[city]
    fp = gpd.read_file(FOOTPRINTS(city))
    fp = fp[fp.geometry.notna() & ~fp.geometry.is_empty]
    src = rasterio.open(NDSM(city)); fp = fp.to_crs(src.crs)
    fp = fp[fp.geometry.area >= MIN_PART_AREA_M2]; fp.sindex      # area measured in the nDSM CRS (metres)
    # CRS guard: the sample points, read in CRS[city], must fall on the city's FOOTPRINTS (they were sampled from them).
    # The nDSM may cover less than the footprints (Sao Paulo): that is reported, not an error.
    pts = np.array([[float(e["x_m"][0]), float(e["y_m"][0])] for e in exs])
    px_, py_ = Transformer.from_crs(crs, src.crs, always_xy=True).transform(pts[:, 0], pts[:, 1])
    inbox = lambda bb: (px_ >= bb[0]) & (px_ <= bb[2]) & (py_ >= bb[1]) & (py_ <= bb[3])
    on_fp, on_ndsm = inbox(fp.total_bounds).mean(), inbox(src.bounds).mean()
    if on_fp < 0.9:
        raise ValueError(f"{city}: only {on_fp:.0%} of the points fall inside the footprints' extent when read as {crs}; "
                         f"wrong UTM zone in CRS? (check the CRS the GEE exporter used for this city)")
    if on_ndsm < 0.99:
        print(f"  NOTE {city}: {1 - on_ndsm:.0%} of the points lie outside the nDSM extent; their patches get no dense "
              f"targets near the point (y = -9999 there), the scalar label height_m is unaffected")
    rows, cmp_ = [], []
    ctrl = set(pd.Series(range(len(exs))).sample(min(6, len(exs)), random_state=0))
    tmp = out_path + ".tmp"
    with tf.io.TFRecordWriter(tmp, GZ) as wtr:
        for i, ex in enumerate(exs):
            x = np.stack([ex[n] for n in NAMES]).reshape(len(NAMES), S, S)
            if FLIP_Y: x = x[:, ::-1, :]
            xm, ym, h = float(ex["x_m"][0]), float(ex["y_m"][0]), float(ex["height_m"][0])
            sid = ex["sample_id"][0].decode()
            y = target_layers(src, fp, xm, ym, crs)
            near = y[1, R - 5:R + 6, R - 5:R + 6]; near = near[near != NODATA]
            cmp_.append((h, near.mean() if len(near) else np.nan, int(ex["is_background"][0]), split_of[sid]))
            wtr.write(tf.train.Example(features=tf.train.Features(feature={
                "x": fl(x), "y": fl(y), "height_m": fl(h), "is_background": fl(ex["is_background"][0]),
                "x_m": fl(xm), "y_m": fl(ym),
                "block_x": fl(np.floor(xm / BLOCK_M)), "block_y": fl(np.floor(ym / BLOCK_M)),
                "sample_id": by(ex["sample_id"][0]), "city": by(ex["city"][0]),
                "split": by(split_of[sid].encode())})).SerializeToString())
            if i in ctrl:
                rows.append((sid, xm, ym, h, x, y))
            if (i + 1) % 500 == 0:
                print(f"    {i + 1} / {len(exs)}")
    os.replace(tmp, out_path)
    src.close()
    c = pd.DataFrame(cmp_, columns=["label", "ndsm_bldg_near", "bg", "split"]); nb = c[c.bg == 0].dropna()
    print(f"  {len(exs)} patches -> {out_path} | split {c.split.value_counts().to_dict()}")
    print(f"  label vs nDSM building height within 50 m: corr {nb.label.corr(nb.ndsm_bldg_near):.2f}, "
          f"median nDSM/label {np.median(nb.ndsm_bldg_near / nb.label):.2f}, patches without nDSM building cells near the point: "
          f"{int(c[c.bg == 0].ndsm_bldg_near.isna().sum())}")
    os.makedirs(os.path.join(OUT, "control_tifs"), exist_ok=True)
    for sid, xm, ym, h, xx, yy in rows:          # 3 targets + S3 red/green/blue, for QGIS
        w_, s_, e_, n_ = corners(xm, ym)
        stack = np.concatenate([yy, xx[[NAMES.index("S3_B4"), NAMES.index("S3_B3"), NAMES.index("S3_B2")]]])
        with rasterio.open(f"{OUT}/control_tifs/{sid}_h{h:.0f}m.tif", "w", driver="GTiff", count=6, dtype="float32",
                           height=S, width=S, crs=crs, transform=from_origin(w_, n_, PX, PX), nodata=NODATA) as dst:
            dst.write(stack)
            for k, nme in enumerate(TARGETS + ["S3_B4", "S3_B3", "S3_B2"], 1): dst.set_band_description(k, nme)
    print(f"  wrote {len(rows)} QGIS GeoTIFFs to {OUT}/control_tifs/")


os.makedirs(OUT, exist_ok=True)
for city in sorted(EXPECTED):
    out_path = os.path.join(OUT, FNAME(city))
    if os.path.exists(out_path) and city not in REBUILD:
        relabel(city, out_path)
    else:
        build(city, out_path)

# ---- final count per city and split, from the written files ----
cnt = []
for city in sorted(EXPECTED):
    p = os.path.join(OUT, FNAME(city))
    if not os.path.exists(p):
        continue
    for raw in tf.data.TFRecordDataset([p], compression_type="GZIP").as_numpy_iterator():
        ex = tf.train.Example.FromString(raw)
        cnt.append((city, ex.features.feature["split"].bytes_list.value[0].decode()))
tab = pd.DataFrame(cnt, columns=["city", "split"]).value_counts().unstack(fill_value=0)
tab.to_csv(os.path.join(OUT, "patch_counts.csv"))
print(f"\nrecords per city and split in {OUT}:\n{tab}")

## QA
Run after both sets are written; the loader below reads the training files.

In [ ]:
# QA of the written patches: missing-data share per season, valid share per target, height summary.
# Loader as in docs/README_patch_dataset.md, with the full 33 x 33 patch (PATCH_PX = 33).
import glob, tensorflow as tf
C_IN, C_OUT, S, NODATA = 48, 3, 33, -9999.0
spec = {"x": tf.io.FixedLenFeature([C_IN * S * S], tf.float32), "y": tf.io.FixedLenFeature([C_OUT * S * S], tf.float32),
        "height_m": tf.io.FixedLenFeature([1], tf.float32), "is_background": tf.io.FixedLenFeature([1], tf.float32),
        "x_m": tf.io.FixedLenFeature([1], tf.float32), "y_m": tf.io.FixedLenFeature([1], tf.float32),
        "block_x": tf.io.FixedLenFeature([1], tf.float32), "block_y": tf.io.FixedLenFeature([1], tf.float32),
        "sample_id": tf.io.FixedLenFeature([1], tf.string), "city": tf.io.FixedLenFeature([1], tf.string),
        "split": tf.io.FixedLenFeature([1], tf.string)}

def parse(rec):
    e = tf.io.parse_single_example(rec, spec)
    return {"x": tf.transpose(tf.reshape(e["x"], [C_IN, S, S]), [1, 2, 0]),      # (33, 33, 48), channels last
            "y": tf.transpose(tf.reshape(e["y"], [C_OUT, S, S]), [1, 2, 0]),     # (33, 33, 3)
            "height_m": e["height_m"][0], "city": e["city"][0], "sample_id": e["sample_id"][0], "split": e["split"][0]}

files = sorted(glob.glob(os.path.join(OUT, "*_dataset.tfrecord.gz")))
ds = tf.data.TFRecordDataset(files, compression_type="GZIP").map(parse)

import numpy as np, pandas as pd
rows = []
for b in ds.batch(512):
    x = b["x"].numpy(); y = b["y"].numpy()
    c = [s.decode() for s in b["city"].numpy()]
    nod_x = (x == NODATA).reshape(len(c), 33*33, 4, 12).mean(axis=(1, 3))   # share per season
    valid_y = (y != NODATA).mean(axis=(1, 2))                                # share per target channel
    for i in range(len(c)):
        rows.append([c[i], float(b["height_m"][i]), *nod_x[i], *valid_y[i]])
df = pd.DataFrame(rows, columns=["city", "h", "nod_S1", "nod_S2", "nod_S3", "nod_S4",
                                 "ok_ndsm", "ok_bldg_h", "ok_frac"])
print(df.groupby("city")[["nod_S1", "nod_S2", "nod_S3", "nod_S4", "ok_ndsm", "ok_bldg_h", "ok_frac"]].mean().round(3))
print(df.groupby("city").h.describe().round(1))
print("patches with any season >50% missing:", (df[["nod_S1","nod_S2","nod_S3","nod_S4"]].max(axis=1) > 0.5).groupby(df.city).sum().to_dict())